In [1]:
# Same as ver03 in terms of codes, but the whole data is updated

In [2]:
# Repository setup and shared helper code
import os
import sys
import runpy
from pathlib import Path

import joblib
import keras
import math
import numpy as np
import pandas as pd

possible_roots = [Path.cwd(), *Path.cwd().parents]
root_path = next((base.resolve() for base in possible_roots if (base / "src").exists()), None)
if root_path is None:
    raise FileNotFoundError("Could not locate the repository root containing src/")
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

from src.paths import MGT_MODEL_DIR, OFFSHORE_MICROGRID_DATA_DIR, output_path, require_path

src_dir = root_path / "src"
for source_file in [
    "plotting.py",
    "thermodynamic_model_functions.py",
    "design_data.py",
    "components_definition.py",
    "components_coefficients.py",
    "experimental_data_post_process_functions.py",
    "experimental_data_post_process_functions_psi_data.py",
    "functions_for_offshore_microgrid_optimization_im_gf_ver02.py",
]:
    code = (src_dir / source_file).read_text(encoding="utf-8")
    exec(compile(code, str(src_dir / source_file), "exec"), globals())


In [3]:
name = "Offshore Microgrid Optimization - IM - GF - Day 3"

main_path = str(output_path("models", name))
path_training_history = str(output_path("models", name, "Training History"))
path_trained_model_and_scaler = str(output_path("models", name, "Trained Model"))
path_saving_figure = str(output_path("figures", name))
path_saving_data_results = str(output_path("results", name))


In [4]:
# Load required week and operation data from repository artifacts
week_data_path = require_path(OFFSHORE_MICROGRID_DATA_DIR / "integrated_data_complete.xlsx")
week_data = pd.read_excel(week_data_path, sheet_name="week 1")

GFA_operation_df_path = require_path(OFFSHORE_MICROGRID_DATA_DIR / "GFA_operation_df.pkl")
GFA_operation_df = pd.read_pickle(GFA_operation_df_path)

GFC_operation_df_path = require_path(OFFSHORE_MICROGRID_DATA_DIR / "GFC_operation_df.pkl")
GFC_operation_df = pd.read_pickle(GFC_operation_df_path)

globals().update(runpy.run_path(str(src_dir / "offshore_microgrid_optimization_gfs_data_generation.py"), init_globals=globals()))


2026-05-29 11:13:46.511998: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
week_data['P_net [MW]'] = week_data['P_prod_WT [MW]']-(week_data['P_dem_GFA [MW]']+week_data['P_dem_GFB [MW]']+week_data['P_dem_GFC [MW]'])


plot_params = ['P_prod_WT [MW]', 'P_dem_GFA [MW]', 'Q_dem_GFA [MW]', 'P_dem_GFB [MW]',
       'Q_dem_GFB [MW]', 'P_dem_GFC [MW]', 'Q_dem_GFC [MW]','P_net [MW]']

plot_params = ['P_net [MW]']

for day in range(7):
    str_row = day*24
    end_row = (day+1)*24
    print('Day:',day)
    for param in plot_params:
        simple_plotter(np.arange(24), [week_data[param].iloc[str_row:end_row]],y_label=param)





Day: 0
Day: 1
Day: 2
Day: 3


Day: 4
Day: 5
Day: 6


In [6]:
GFC_operation_df = GFC_operation_df[(GFC_operation_df['T_amb_GFC [degC]']<week_data['T_amb [degC]'].max()+1)&
                (GFC_operation_df['T_amb_GFC [degC]']>week_data['T_amb [degC]'].min()-1)]


GFA_operation_df = GFA_operation_df[(GFA_operation_df['T_amb_GFA [degC]']<week_data['T_amb [degC]'].max()+1)&
                (GFA_operation_df['T_amb_GFA [degC]']>week_data['T_amb [degC]'].min()-1)]


In [7]:
# Load MGT model artifacts from repository artifacts
scaler_MGT = joblib.load(require_path(MGT_MODEL_DIR / "model1_scaler.save"))
model_MGT = keras.models.load_model(require_path(MGT_MODEL_DIR / "model1.h5"))
input_parameters_MGT = ["P_ref [kW]", "T1 [degC]"]
output_parameters_MGT = ["m_f [gr/s]", "TOT [degC]", "N_act [rpm]", "P_act [kW]"]


In [8]:
# resizing the elements so we have a conceptual MG with good sizing
week_data_reserve = week_data.copy()
df = week_data_reserve.copy()


week_data_reserve = week_data.copy()
df = week_data_reserve.copy()



In [9]:
row_begin_day_1 = 0 * 24
row_end_day_1 = row_begin_day_1 + 24
df_study_day_1 = df.iloc[row_begin_day_1:row_end_day_1].copy()

row_begin_day_2 = 1 * 24
row_end_day_2 = row_begin_day_2 + 24
df_study_day_2 = df.iloc[row_begin_day_2:row_end_day_2].copy()

row_begin_day_3 = 2 * 24
row_end_day_3 = row_begin_day_3 + 24
df_study_day_3 = df.iloc[row_begin_day_3:row_end_day_3].copy()

row_begin_day_4 = 3 * 24
row_end_day_4 = row_begin_day_4 + 24
df_study_day_4 = df.iloc[row_begin_day_4:row_end_day_4].copy()

row_begin_day_5 = 4 * 24
row_end_day_5 = row_begin_day_5 + 24
df_study_day_5 = df.iloc[row_begin_day_5:row_end_day_5].copy()

row_begin_day_6 = 5 * 24
row_end_day_6 = row_begin_day_6 + 24
df_study_day_6 = df.iloc[row_begin_day_6:row_end_day_6].copy()

row_begin_day_7 = 6 * 24
row_end_day_7 = row_begin_day_7 + 24
df_study_day_7 = df.iloc[row_begin_day_7:row_end_day_7].copy()

## Optimization

In [10]:
df = df_study_day_3

In [11]:
observation_window = 6
delta_t_time_step = 3600

observation_window_daily = 24

In [12]:
# optimization parameters with [min, max, initial guess]


opt_param_dict = {
'P_GFA_to_GFC':[-20,20,10],
'P_GFA_to_ELZ':[0,100,0],
'FR_GFA_GT1':[0.516,1,1],
# 'FR_GFA_GT2':[0,1,1],
# 'FR_GFA_GT3':[0,1,1],
# 'FR_GFA_GT4':[0.516,1,1],
}

x_band, x_band_statements, parameter_definition, x_initial, no_initial_pop, initial_pop = MG_parameter_boundary_def(opt_param_dict, observation_window)



In [13]:
sell=False
buy=False
hours=6
day_hours = 24

In [14]:
reserve_H2_cb = 0* np.ones(round(7*24/hours)+1)
reserve_H2_op = 0* np.ones(round(7*24/hours)+1)


In [15]:
reserve_H2_cb[0] = 2.417
reserve_H2_op[0] = 1.228


In [16]:
display_step=1000

In [17]:
period = 1

reserve_H2_cb_per1 = reserve_H2_cb[period-1]
reserve_H2_op_per1 = reserve_H2_op[period-1]


MG_run_cb_per1, cost_total_cb_per1, penalty_total_cb_per1 = MG_run_cb(df.iloc[(period-1)*hours:(period)*hours], reserve_H2_cb[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',MG_run_cb_per1['cost_total [EUR]'].sum())
x_initial_from_cb = extract_x_span_for_initialization(MG_run_cb_per1, x_band_statements)    
simulation_per1, x_record_per1, error_record_per1 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, x_initial_from_cb, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='Nelder-Mead',sell=sell,buy=buy,display_step=display_step)



Total cost: 41131.432042265544


 
>>>>>> itr =  0  cost =  6276.875
Total cost: 6953.023
Penalty cost: 0.0
NOx emission cost: 458.403
H2 incentive: 676.148
Reserved H2 for the next round: 1.871
Deviations cost: 0
Operation cost: 41718.137
>>>>>>>>>>>>> The optimization is finished.


In [18]:
opt_solution_index_per1 = np.where(error_record_per1 == error_record_per1.min())
opt_solution_index_per1 = opt_solution_index_per1[0][0]
best_simulation_per1 = simulation_per1[opt_solution_index_per1]
best_itr_x_per1 = x_record_per1[opt_solution_index_per1]

display(best_itr_x_per1)

array([ 5.00000000e-01,  7.13398041e-01, -2.41528138e-04,  5.00000000e-01,
        7.89794916e-01, -2.41528138e-04,  5.00000000e-01,  6.46440820e-01,
       -2.41528138e-04,  5.00000000e-01,  7.15867564e-01, -2.41528138e-04,
        5.00000000e-01,  5.98472954e-01, -2.41528138e-04,  5.00000000e-01,
        7.83092040e-01, -2.41528138e-04])

In [19]:
period = 1

simulation_per1, x_record_per1, error_record_per1 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per1, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  6276.875
Total cost: 6953.023
Penalty cost: 0.0
NOx emission cost: 458.403
H2 incentive: 676.148
Reserved H2 for the next round: 1.871
Deviations cost: 0
Operation cost: 41718.137
>>>>>>>>>>>>> The optimization is finished.


In [20]:
opt_solution_index_per1 = np.where(error_record_per1 == error_record_per1.min())
opt_solution_index_per1 = opt_solution_index_per1[0][0]
best_simulation_per1 = simulation_per1[opt_solution_index_per1]
best_itr_x_per1 = x_record_per1[opt_solution_index_per1]

display(best_itr_x_per1)

array([ 5.00000000e-01,  7.13398041e-01, -2.41528138e-04,  5.00000000e-01,
        7.89794916e-01, -2.41528138e-04,  5.00000000e-01,  6.46440820e-01,
       -2.41528138e-04,  5.00000000e-01,  7.15867564e-01, -2.41528138e-04,
        5.00000000e-01,  5.98472954e-01, -2.41528138e-04,  5.00000000e-01,
        7.83092040e-01, -2.41528138e-04])

In [21]:
period = 1

simulation_per1, x_record_per1, error_record_per1 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per1, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  6276.875
Total cost: 6953.023
Penalty cost: 0.0
NOx emission cost: 458.403
H2 incentive: 676.148
Reserved H2 for the next round: 1.871
Deviations cost: 0
Operation cost: 41718.137
>>>>>>>>>>>>> The optimization is finished.


In [22]:
opt_solution_index_per1 = np.where(error_record_per1 == error_record_per1.min())
opt_solution_index_per1 = opt_solution_index_per1[0][0]
best_simulation_per1 = simulation_per1[opt_solution_index_per1]
best_itr_x_per1 = x_record_per1[opt_solution_index_per1]

display(best_itr_x_per1)

array([ 5.00000000e-01,  7.13398041e-01, -2.41528138e-04,  5.00000000e-01,
        7.89794916e-01, -2.41528138e-04,  5.00000000e-01,  6.46440820e-01,
       -2.41528138e-04,  5.00000000e-01,  7.15867564e-01, -2.41528138e-04,
        5.00000000e-01,  5.98472954e-01, -2.41528138e-04,  5.00000000e-01,
        7.83092040e-01, -2.41528138e-04])

In [23]:
period = 1

simulation_per1, x_record_per1, error_record_per1 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per1, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  6276.875
Total cost: 6953.023
Penalty cost: 0.0
NOx emission cost: 458.403
H2 incentive: 676.148
Reserved H2 for the next round: 1.871
Deviations cost: 0
Operation cost: 41718.137
>>>>>>>>>>>>> The optimization is finished.


In [24]:
opt_solution_index_per1 = np.where(error_record_per1 == error_record_per1.min())
opt_solution_index_per1 = opt_solution_index_per1[0][0]
best_simulation_per1 = simulation_per1[opt_solution_index_per1]
best_itr_x_per1 = x_record_per1[opt_solution_index_per1]
print('Total cost:',best_simulation_per1['cost_total [EUR]'].sum())

display(best_itr_x_per1)


Total cost: 41718.13739318335


array([ 5.00000000e-01,  7.13398041e-01, -2.41528138e-04,  5.00000000e-01,
        7.89794916e-01, -2.41528138e-04,  5.00000000e-01,  6.46440820e-01,
       -2.41528138e-04,  5.00000000e-01,  7.15867564e-01, -2.41528138e-04,
        5.00000000e-01,  5.98472954e-01, -2.41528138e-04,  5.00000000e-01,
        7.83092040e-01, -2.41528138e-04])

In [25]:
period = 1

best_simulation_per1, cost_total_avg, penalty_total_avg =  MG_run_evaluation(df.iloc[(period-1)*hours:(period)*hours], best_itr_x_per1, parameter_definition, reserve_H2_op[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',best_simulation_per1['cost_total [EUR]'].sum())



Total cost: 41992.9130879975


In [26]:
for param in MG_run_cb_per1.columns:

    y1 = MG_run_cb_per1[param]
    y2 = best_simulation_per1[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [27]:
reserve_H2_cb[1] = MG_run_cb_per1['H2_tank_aval [kg/s]'].iloc[-1] + MG_run_cb_per1['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period condition-based:', parameter_formater(reserve_H2_cb[1],3))

reserve_H2_op[1] = best_simulation_per1['H2_tank_aval [kg/s]'].iloc[-1] + best_simulation_per1['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period optimization:', parameter_formater(reserve_H2_op[1],3))



H2 Tank content for next period condition-based: 3.511
H2 Tank content for next period optimization: 1.935


In [28]:
period = 2


MG_run_cb_per2, cost_total_cb_per2, penalty_total_cb_per2 = MG_run_cb(df.iloc[(period-1)*hours:(period)*hours], reserve_H2_cb[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',MG_run_cb_per2['cost_total [EUR]'].sum())
x_initial_from_cb = extract_x_span_for_initialization(MG_run_cb_per2, x_band_statements)    
simulation_per2, x_record_per2, error_record_per2 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, x_initial_from_cb, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='Nelder-Mead',sell=sell,buy=buy,display_step=display_step)





Total cost: 38857.139039151014


 
>>>>>> itr =  0  cost =  5695.506
Total cost: 6543.928
Penalty cost: 0.0
NOx emission cost: 454.448
H2 incentive: 848.422
Reserved H2 for the next round: 2.406
Deviations cost: 0
Operation cost: 39263.567
>>>>>>>>>>>>> The optimization is finished.


In [29]:
opt_solution_index_per2 = np.where(error_record_per2 == error_record_per2.min())
opt_solution_index_per2 = opt_solution_index_per2[0][0]
best_simulation_per2 = simulation_per2[opt_solution_index_per2]
best_itr_x_per2 = x_record_per2[opt_solution_index_per2]

display(best_itr_x_per2)

array([ 5.00000000e-01,  6.65584273e-01, -2.41528138e-04,  5.00000000e-01,
        5.63252166e-01, -2.41528138e-04,  5.00000000e-01,  6.59803879e-01,
       -2.41528138e-04,  5.00000000e-01,  6.83038442e-01, -2.41528138e-04,
        5.00000000e-01,  6.49725736e-01, -2.41528138e-04,  5.00000000e-01,
        6.30598490e-01, -2.41528138e-04])

In [30]:
period = 2

simulation_per2, x_record_per2, error_record_per2 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per2, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  5695.506
Total cost: 6543.928
Penalty cost: 0.0
NOx emission cost: 454.448
H2 incentive: 848.422
Reserved H2 for the next round: 2.406
Deviations cost: 0
Operation cost: 39263.567
>>>>>>>>>>>>> The optimization is finished.


In [31]:
opt_solution_index_per2 = np.where(error_record_per2 == error_record_per2.min())
opt_solution_index_per2 = opt_solution_index_per2[0][0]
best_simulation_per2 = simulation_per2[opt_solution_index_per2]
best_itr_x_per2 = x_record_per2[opt_solution_index_per2]

display(best_itr_x_per2)

array([ 5.00000000e-01,  6.65584273e-01, -2.41528138e-04,  5.00000000e-01,
        5.63252166e-01, -2.41528138e-04,  5.00000000e-01,  6.59803879e-01,
       -2.41528138e-04,  5.00000000e-01,  6.83038442e-01, -2.41528138e-04,
        5.00000000e-01,  6.49725736e-01, -2.41528138e-04,  5.00000000e-01,
        6.30598490e-01, -2.41528138e-04])

In [32]:
period = 2

simulation_per2, x_record_per2, error_record_per2 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per2, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  5695.506
Total cost: 6543.928
Penalty cost: 0.0
NOx emission cost: 454.448
H2 incentive: 848.422
Reserved H2 for the next round: 2.406
Deviations cost: 0
Operation cost: 39263.567
>>>>>>>>>>>>> The optimization is finished.


In [33]:
opt_solution_index_per2 = np.where(error_record_per2 == error_record_per2.min())
opt_solution_index_per2 = opt_solution_index_per2[0][0]
best_simulation_per2 = simulation_per2[opt_solution_index_per2]
best_itr_x_per2 = x_record_per2[opt_solution_index_per2]

display(best_itr_x_per2)

array([ 5.00000000e-01,  6.65584273e-01, -2.41528138e-04,  5.00000000e-01,
        5.63252166e-01, -2.41528138e-04,  5.00000000e-01,  6.59803879e-01,
       -2.41528138e-04,  5.00000000e-01,  6.83038442e-01, -2.41528138e-04,
        5.00000000e-01,  6.49725736e-01, -2.41528138e-04,  5.00000000e-01,
        6.30598490e-01, -2.41528138e-04])

In [34]:
period = 2

simulation_per2, x_record_per2, error_record_per2 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per2, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  5695.506
Total cost: 6543.928
Penalty cost: 0.0
NOx emission cost: 454.448
H2 incentive: 848.422
Reserved H2 for the next round: 2.406
Deviations cost: 0
Operation cost: 39263.567
>>>>>>>>>>>>> The optimization is finished.


In [35]:
opt_solution_index_per2 = np.where(error_record_per2 == error_record_per2.min())
opt_solution_index_per2 = opt_solution_index_per2[0][0]
best_simulation_per2 = simulation_per2[opt_solution_index_per2]
best_itr_x_per2 = x_record_per2[opt_solution_index_per2]
print('Total cost:',best_simulation_per2['cost_total [EUR]'].sum())
display(best_itr_x_per2)

Total cost: 39263.56715910444


array([ 5.00000000e-01,  6.65584273e-01, -2.41528138e-04,  5.00000000e-01,
        5.63252166e-01, -2.41528138e-04,  5.00000000e-01,  6.59803879e-01,
       -2.41528138e-04,  5.00000000e-01,  6.83038442e-01, -2.41528138e-04,
        5.00000000e-01,  6.49725736e-01, -2.41528138e-04,  5.00000000e-01,
        6.30598490e-01, -2.41528138e-04])

In [36]:
period = 2

best_simulation_per2, cost_total_avg, penalty_total_avg =  MG_run_evaluation(df.iloc[(period-1)*hours:(period)*hours], best_itr_x_per2, parameter_definition, reserve_H2_op[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',best_simulation_per2['cost_total [EUR]'].sum())



Total cost: 39772.51571990115


In [37]:
for param in MG_run_cb_per2.columns:

    y1 = MG_run_cb_per2[param]
    y2 = best_simulation_per2[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [38]:
reserve_H2_cb[2] = MG_run_cb_per2['H2_tank_aval [kg/s]'].iloc[-1] + MG_run_cb_per2['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period condition-based:', parameter_formater(reserve_H2_cb[2],3))

reserve_H2_op[2] = best_simulation_per2['H2_tank_aval [kg/s]'].iloc[-1] + best_simulation_per2['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period optimization:', parameter_formater(reserve_H2_op[2],3))



H2 Tank content for next period condition-based: 4.441
H2 Tank content for next period optimization: 2.569


In [39]:
period = 3


MG_run_cb_per3, cost_total_cb_per3, penalty_total_cb_per3 = MG_run_cb(df.iloc[(period-1)*hours:(period)*hours], reserve_H2_cb[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',MG_run_cb_per3['cost_total [EUR]'].sum())
x_initial_from_cb = extract_x_span_for_initialization(MG_run_cb_per3, x_band_statements)    
simulation_per3, x_record_per3, error_record_per3 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, x_initial_from_cb, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='Nelder-Mead',sell=sell,buy=buy,display_step=display_step)





Total cost: 39189.16849874687


 
>>>>>> itr =  0  cost =  5382.824
Total cost: 6468.497
Penalty cost: 0.0
NOx emission cost: 441.765
H2 incentive: 1085.673
Reserved H2 for the next round: 3.252
Deviations cost: 0
Operation cost: 38810.984
>>>>>>>>>>>>> The optimization is finished.


In [40]:
opt_solution_index_per3 = np.where(error_record_per3 == error_record_per3.min())
opt_solution_index_per3 = opt_solution_index_per3[0][0]
best_simulation_per3 = simulation_per3[opt_solution_index_per3]
best_itr_x_per3 = x_record_per3[opt_solution_index_per3]

display(best_itr_x_per3)

array([ 5.00000000e-01,  7.38400270e-01, -2.41528138e-04,  5.00000000e-01,
        7.14799822e-01, -2.41528138e-04,  5.00000000e-01,  7.33556243e-01,
       -2.41528138e-04,  5.00000000e-01,  7.62662790e-01, -2.41528138e-04,
        5.00000000e-01,  4.01294840e-01, -2.41528138e-04,  5.00000000e-01,
        3.83731798e-01, -2.41528138e-04])

In [41]:
period = 3

simulation_per3, x_record_per3, error_record_per3 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per3, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  5382.824
Total cost: 6468.497
Penalty cost: 0.0
NOx emission cost: 441.765
H2 incentive: 1085.673
Reserved H2 for the next round: 3.252
Deviations cost: 0
Operation cost: 38810.984
>>>>>>>>>>>>> The optimization is finished.


In [42]:
opt_solution_index_per3 = np.where(error_record_per3 == error_record_per3.min())
opt_solution_index_per3 = opt_solution_index_per3[0][0]
best_simulation_per3 = simulation_per3[opt_solution_index_per3]
best_itr_x_per3 = x_record_per3[opt_solution_index_per3]

display(best_itr_x_per3)

array([ 5.00000000e-01,  7.38400270e-01, -2.41528138e-04,  5.00000000e-01,
        7.14799822e-01, -2.41528138e-04,  5.00000000e-01,  7.33556243e-01,
       -2.41528138e-04,  5.00000000e-01,  7.62662790e-01, -2.41528138e-04,
        5.00000000e-01,  4.01294840e-01, -2.41528138e-04,  5.00000000e-01,
        3.83731798e-01, -2.41528138e-04])

In [43]:
period = 3

simulation_per3, x_record_per3, error_record_per3 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per3, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  5382.824
Total cost: 6468.497
Penalty cost: 0.0
NOx emission cost: 441.765
H2 incentive: 1085.673
Reserved H2 for the next round: 3.252
Deviations cost: 0
Operation cost: 38810.984
>>>>>>>>>>>>> The optimization is finished.


In [44]:
opt_solution_index_per3 = np.where(error_record_per3 == error_record_per3.min())
opt_solution_index_per3 = opt_solution_index_per3[0][0]
best_simulation_per3 = simulation_per3[opt_solution_index_per3]
best_itr_x_per3 = x_record_per3[opt_solution_index_per3]

display(best_itr_x_per3)

array([ 5.00000000e-01,  7.38400270e-01, -2.41528138e-04,  5.00000000e-01,
        7.14799822e-01, -2.41528138e-04,  5.00000000e-01,  7.33556243e-01,
       -2.41528138e-04,  5.00000000e-01,  7.62662790e-01, -2.41528138e-04,
        5.00000000e-01,  4.01294840e-01, -2.41528138e-04,  5.00000000e-01,
        3.83731798e-01, -2.41528138e-04])

In [45]:
period = 3

simulation_per3, x_record_per3, error_record_per3 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per3, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  5382.824
Total cost: 6468.497
Penalty cost: 0.0
NOx emission cost: 441.765
H2 incentive: 1085.673
Reserved H2 for the next round: 3.252
Deviations cost: 0
Operation cost: 38810.984
>>>>>>>>>>>>> The optimization is finished.


In [46]:
opt_solution_index_per3 = np.where(error_record_per3 == error_record_per3.min())
opt_solution_index_per3 = opt_solution_index_per3[0][0]
best_simulation_per3 = simulation_per3[opt_solution_index_per3]
best_itr_x_per3 = x_record_per3[opt_solution_index_per3]
print('Total cost:',best_simulation_per3['cost_total [EUR]'].sum())

display(best_itr_x_per3)

Total cost: 38810.984406977404


array([ 5.00000000e-01,  7.38400270e-01, -2.41528138e-04,  5.00000000e-01,
        7.14799822e-01, -2.41528138e-04,  5.00000000e-01,  7.33556243e-01,
       -2.41528138e-04,  5.00000000e-01,  7.62662790e-01, -2.41528138e-04,
        5.00000000e-01,  4.01294840e-01, -2.41528138e-04,  5.00000000e-01,
        3.83731798e-01, -2.41528138e-04])

In [47]:
period = 3

best_simulation_per3, cost_total_avg, penalty_total_avg =  MG_run_evaluation(df.iloc[(period-1)*hours:(period)*hours], best_itr_x_per3, parameter_definition, reserve_H2_op[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',best_simulation_per3['cost_total [EUR]'].sum())



Total cost: 38225.55874593464


In [48]:
for param in MG_run_cb_per3.columns:

    y1 = MG_run_cb_per3[param]
    y2 = best_simulation_per3[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [49]:
reserve_H2_cb[3] = MG_run_cb_per3['H2_tank_aval [kg/s]'].iloc[-1] + MG_run_cb_per3['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period condition-based:', parameter_formater(reserve_H2_cb[3],3))

reserve_H2_op[3] = best_simulation_per3['H2_tank_aval [kg/s]'].iloc[-1] + best_simulation_per3['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period optimization:', parameter_formater(reserve_H2_op[3],3))



H2 Tank content for next period condition-based: 5.323
H2 Tank content for next period optimization: 3.421


In [50]:
period = 4


MG_run_cb_per4, cost_total_cb_per4, penalty_total_cb_per4 = MG_run_cb(df.iloc[(period-1)*hours:(period)*hours], reserve_H2_cb[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',MG_run_cb_per4['cost_total [EUR]'].sum())
x_initial_from_cb = extract_x_span_for_initialization(MG_run_cb_per4, x_band_statements)    
simulation_per4, x_record_per4, error_record_per4 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, x_initial_from_cb, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='Nelder-Mead',sell=sell,buy=buy,display_step=display_step)





Total cost: 38765.0126477942


 
>>>>>> itr =  0  cost =  4876.38
Total cost: 6294.557
Penalty cost: 0.0
NOx emission cost: 424.396
H2 incentive: 1418.176
Reserved H2 for the next round: 4.103
Deviations cost: 0
Operation cost: 37767.34
>>>>>>>>>>>>> The optimization is finished.


In [51]:
opt_solution_index_per4 = np.where(error_record_per4 == error_record_per4.min())
opt_solution_index_per4 = opt_solution_index_per4[0][0]
best_simulation_per4 = simulation_per4[opt_solution_index_per4]
best_itr_x_per4 = x_record_per4[opt_solution_index_per4]

display(best_itr_x_per4)

array([ 5.00000000e-01,  4.60198893e-01, -2.41528138e-04,  5.00000000e-01,
        5.67653948e-01, -2.41528138e-04,  5.00000000e-01,  6.03619921e-01,
       -2.41528138e-04,  5.00000000e-01,  7.70776011e-01, -2.41528138e-04,
        5.00000000e-01,  6.74557218e-01, -2.41528138e-04,  5.00000000e-01,
        6.32196304e-01, -2.41528138e-04])

In [52]:
period = 4

simulation_per4, x_record_per4, error_record_per4 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per4, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  4876.38
Total cost: 6294.557
Penalty cost: 0.0
NOx emission cost: 424.396
H2 incentive: 1418.176
Reserved H2 for the next round: 4.103
Deviations cost: 0
Operation cost: 37767.34
>>>>>>>>>>>>> The optimization is finished.


In [53]:
opt_solution_index_per4 = np.where(error_record_per4 == error_record_per4.min())
opt_solution_index_per4 = opt_solution_index_per4[0][0]
best_simulation_per4 = simulation_per4[opt_solution_index_per4]
best_itr_x_per4 = x_record_per4[opt_solution_index_per4]

display(best_itr_x_per4)

array([ 5.00000000e-01,  4.60198893e-01, -2.41528138e-04,  5.00000000e-01,
        5.67653948e-01, -2.41528138e-04,  5.00000000e-01,  6.03619921e-01,
       -2.41528138e-04,  5.00000000e-01,  7.70776011e-01, -2.41528138e-04,
        5.00000000e-01,  6.74557218e-01, -2.41528138e-04,  5.00000000e-01,
        6.32196304e-01, -2.41528138e-04])

In [54]:
period = 4

simulation_per4, x_record_per4, error_record_per4 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per4, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  4876.38
Total cost: 6294.557
Penalty cost: 0.0
NOx emission cost: 424.396
H2 incentive: 1418.176
Reserved H2 for the next round: 4.103
Deviations cost: 0
Operation cost: 37767.34
>>>>>>>>>>>>> The optimization is finished.


In [55]:
opt_solution_index_per4 = np.where(error_record_per4 == error_record_per4.min())
opt_solution_index_per4 = opt_solution_index_per4[0][0]
best_simulation_per4 = simulation_per4[opt_solution_index_per4]
best_itr_x_per4 = x_record_per4[opt_solution_index_per4]

display(best_itr_x_per4)

array([ 5.00000000e-01,  4.60198893e-01, -2.41528138e-04,  5.00000000e-01,
        5.67653948e-01, -2.41528138e-04,  5.00000000e-01,  6.03619921e-01,
       -2.41528138e-04,  5.00000000e-01,  7.70776011e-01, -2.41528138e-04,
        5.00000000e-01,  6.74557218e-01, -2.41528138e-04,  5.00000000e-01,
        6.32196304e-01, -2.41528138e-04])

In [56]:
period = 4

simulation_per4, x_record_per4, error_record_per4 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per4, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  4876.38
Total cost: 6294.557
Penalty cost: 0.0
NOx emission cost: 424.396
H2 incentive: 1418.176
Reserved H2 for the next round: 4.103
Deviations cost: 0
Operation cost: 37767.34
>>>>>>>>>>>>> The optimization is finished.


In [57]:
opt_solution_index_per4 = np.where(error_record_per4 == error_record_per4.min())
opt_solution_index_per4 = opt_solution_index_per4[0][0]
best_simulation_per4 = simulation_per4[opt_solution_index_per4]
best_itr_x_per4 = x_record_per4[opt_solution_index_per4]
print('Total cost:',best_simulation_per4['cost_total [EUR]'].sum())

display(best_itr_x_per4)

Total cost: 37767.34048137437


array([ 5.00000000e-01,  4.60198893e-01, -2.41528138e-04,  5.00000000e-01,
        5.67653948e-01, -2.41528138e-04,  5.00000000e-01,  6.03619921e-01,
       -2.41528138e-04,  5.00000000e-01,  7.70776011e-01, -2.41528138e-04,
        5.00000000e-01,  6.74557218e-01, -2.41528138e-04,  5.00000000e-01,
        6.32196304e-01, -2.41528138e-04])

In [58]:
 period = 4

best_simulation_per4, cost_total_avg, penalty_total_avg =  MG_run_evaluation(df.iloc[(period-1)*hours:(period)*hours], best_itr_x_per4, parameter_definition, reserve_H2_op[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',best_simulation_per4['cost_total [EUR]'].sum())



Total cost: 37730.86989400336


In [59]:
for param in MG_run_cb_per4.columns:

    y1 = MG_run_cb_per4[param]
    y2 = best_simulation_per4[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [60]:
reserve_H2_cb[4] = MG_run_cb_per4['H2_tank_aval [kg/s]'].iloc[-1] + MG_run_cb_per4['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period condition-based:', parameter_formater(reserve_H2_cb[4],3))

reserve_H2_op[4] = best_simulation_per4['H2_tank_aval [kg/s]'].iloc[-1] + best_simulation_per4['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period optimization:', parameter_formater(reserve_H2_op[4],3))



H2 Tank content for next period condition-based: 6.194
H2 Tank content for next period optimization: 4.274


In [61]:
best_simulation_day1 = pd.concat((best_simulation_per1,best_simulation_per2,best_simulation_per3,best_simulation_per4),axis=0).reset_index(drop=True)
MG_run_cb_day1 = pd.concat((MG_run_cb_per1,MG_run_cb_per2,MG_run_cb_per3,MG_run_cb_per4),axis=0).reset_index(drop=True)




In [62]:
for param in MG_run_cb_day1.columns:

    y1 = MG_run_cb_day1[param]
    y2 = best_simulation_day1[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [63]:
# optimization parameters with [min, max, initial guess]


opt_param_dict_daily = {
'P_GFA_to_GFC':[-20,20,10],
'P_GFA_to_ELZ':[0,100,0],
'FR_GFA_GT1':[0.516,1,1],
# 'FR_GFA_GT2':[0,1,1],
# 'FR_GFA_GT3':[0,1,1],
# 'FR_GFA_GT4':[0.516,1,1],
}

x_band_daily, x_band_statements_daily, parameter_definition_daily, x_initial_daily, no_initial_pop_daily, initial_pop_daily = MG_parameter_boundary_def(opt_param_dict_daily, observation_window_daily)



In [64]:
day = 1
period = 1

MG_run_cb_day1, cost_total_cb_day1, penalty_total_cb_day1 = MG_run_cb(df.iloc[(day-1)*hours:(day)*day_hours], reserve_H2_cb[period-1], x_band_statements_daily, buy=buy, sell=sell)
print('Total cost:',MG_run_cb_day1['cost_total [EUR]'].sum())

x_initial_from_day1 = extract_x_span_for_initialization(best_simulation_day1, x_band_statements_daily)    
simulation_day1, x_record_day1, error_record_day1 = optimize_microgrid_performance(df.iloc[(day-1)*hours:(day)*day_hours], parameter_definition_daily, x_initial_from_day1, x_band_statements, x_band_statements_daily, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='Nelder-Mead',sell=sell,buy=buy,display_step=display_step)





Total cost: 157942.75222795762


 
>>>>>> itr =  0  cost =  6220.657
Total cost: 6565.001
Penalty cost: 0.0
NOx emission cost: 1779.011
H2 incentive: 344.344
Reserved H2 for the next round: 3.981
Deviations cost: 0
Operation cost: 157560.029
>>>>>>>>>>>>> The optimization is finished.


In [65]:
opt_solution_index_day1 = np.where(error_record_day1 == error_record_day1.min())
opt_solution_index_day1 = opt_solution_index_day1[0][0]
best_simulation_day1 = simulation_day1[opt_solution_index_day1]
best_itr_x_day1 = x_record_day1[opt_solution_index_day1]
print('Total cost:',best_simulation_day1['cost_total [EUR]'].sum())

display(best_itr_x_day1)

Total cost: 157560.02944063957


array([ 5.00000000e-01,  7.13398041e-01, -2.41528138e-04,  5.00000000e-01,
        7.89794916e-01, -2.41528138e-04,  5.00000000e-01,  6.46440820e-01,
       -2.41528138e-04,  5.00000000e-01,  7.15867564e-01, -2.41528138e-04,
        5.00000000e-01,  5.98472954e-01, -2.41528138e-04,  5.00000000e-01,
        7.83092040e-01, -2.41528138e-04,  5.00000000e-01,  6.65584273e-01,
       -2.41528138e-04,  5.00000000e-01,  5.63252166e-01, -2.41528138e-04,
        5.00000000e-01,  6.59803879e-01, -2.41528138e-04,  5.00000000e-01,
        6.83038442e-01, -2.41528138e-04,  5.00000000e-01,  6.49725736e-01,
       -2.41528138e-04,  5.00000000e-01,  6.30598490e-01, -2.41528138e-04,
        5.00000000e-01,  7.38400270e-01, -2.41528138e-04,  5.00000000e-01,
        7.14799822e-01, -2.41528138e-04,  5.00000000e-01,  7.33556243e-01,
       -2.41528138e-04,  5.00000000e-01,  7.62662790e-01, -2.41528138e-04,
        5.00000000e-01,  4.01294840e-01, -2.41528138e-04,  5.00000000e-01,
        3.83731798e-01, -

In [66]:
print('H2 Tank content for next period condition-based:', parameter_formater(MG_run_cb_day1['H2_tank_aval [kg/s]'].iloc[-1]+MG_run_cb_day1['H2_tank_var [kg/s]'].iloc[-1],3))

print('H2 Tank content for next period optimization:', parameter_formater(best_simulation_day1['H2_tank_aval [kg/s]'].iloc[-1]+best_simulation_day1['H2_tank_var [kg/s]'].iloc[-1],3))


H2 Tank content for next period condition-based: 6.194
H2 Tank content for next period optimization: 4.117


In [67]:
day = 1
period = 1

best_simulation_day1, cost_total_avg, penalty_total_avg =  MG_run_evaluation(df.iloc[(day-1)*hours:(day)*day_hours], best_itr_x_day1, parameter_definition_daily, reserve_H2_op[period-1], x_band_statements_daily, buy=buy, sell=sell)
print('Total cost:',best_simulation_day1['cost_total [EUR]'].sum())



Total cost: 157721.85744783664


In [68]:
for param in MG_run_cb_day1.columns:

    y1 = MG_run_cb_day1[param]
    y2 = best_simulation_day1[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [69]:
save_df_excel_to_path(MG_run_cb_day1,'performance_day 3','condition-based',path_saving_data_results)
save_df_excel_to_path(best_simulation_day1,'performance_day 3','optimized',path_saving_data_results)


The data-frame is saved in: /Users/2923185/Desktop/Offshore Microgrid clean/Codes/outputs/results/Offshore Microgrid Optimization - IM - GF - Day 3/performance_day 3.xlsx
The data-frame is saved in: /Users/2923185/Desktop/Offshore Microgrid clean/Codes/outputs/results/Offshore Microgrid Optimization - IM - GF - Day 3/performance_day 3.xlsx
